In [1]:
from pathlib import Path
import os,json,hashlib,time,re
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import numpy as np
import pandas as pd
import h5py,torch
from torch import nn
from sklearn.metrics import roc_auc_score,average_precision_score
ROOT=Path(r'C:/Users/choud/Desktop/Minor project/cnn/cnn wave/eew_causal_v3')
RUN=ROOT/'cnn_eew_training';OUT=RUN/'codex_performance_audit'
torch.set_num_threads(4);torch.use_deterministic_algorithms(True);torch.backends.cudnn.benchmark=False
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(4194304),b''):h.update(b)
 return h.hexdigest()
files=[RUN/'final_cnn_eew_report.md',RUN/'01_cnn_training.ipynb',RUN/'02_cnn_evaluation.ipynb',ROOT/'configs/split_manifest.csv',ROOT/'codex_final_verification/hdf5_training_readiness.md']+[ROOT/'dataset'/f'eew_dataset_{s}.h5' for s in ['train','validation','test']]+[RUN/'models'/f'cnn_{h}s_best.pt' for h in [1,2,3]]
before={str(p):sha(p) for p in files}
R={'date':'2026-11-13','checks':{},'input_sha256':before,'scope':'Frozen inference and targeted metadata/pre-P checks; no training, full dataset audit or simulator execution.'}
def check(k,ok,evidence):R['checks'][k]={'status':'PASS' if ok else 'FAIL','evidence':evidence}
# Reconstructed independently from inspected architecture; no training/evaluation cells executed.
class AuditCNN(nn.Module):
 def __init__(self):
  super().__init__();self.features=nn.Sequential(nn.Conv1d(1,16,9,2,4),nn.SiLU(),nn.Conv1d(16,32,7,2,3),nn.SiLU(),nn.Conv1d(32,48,5,2,2),nn.SiLU());self.shared=nn.Sequential(nn.Linear(96,32),nn.SiLU());self.detection=nn.Linear(32,1);self.magnitude=nn.Linear(32,1)
 def forward(self,x):
  u=self.features(x.sign()*x.abs().log1p());v=self.shared(torch.cat((u.mean(2),u.amax(2)),1));return self.detection(v)[:,0],self.magnitude(v)[:,0]
@torch.inference_mode()
def predict(model,x):
 a=[];b=[]
 for i in range(0,len(x),256):
  l,m=model(torch.from_numpy(x[i:i+256,None,:]).to(DEVICE));a.append(l.cpu());b.append(m.cpu())
 return torch.cat(a),torch.cat(b)
def metrics(y,p,m,t,reg):
 y=np.asarray(y,int);pred=np.asarray(p)>=.5;tp=int(((y==1)&pred).sum());tn=int(((y==0)&~pred).sum());fp=int(((y==0)&pred).sum());fn=int(((y==1)&~pred).sum())
 e=np.asarray(m,dtype=np.float64)[reg]-np.asarray(t,dtype=np.float64)[reg];a=np.asarray(t,dtype=np.float64)[reg]
 return {'n':len(y),'earthquakes':int(reg.sum()),'accuracy':(tp+tn)/len(y),'precision':tp/(tp+fp) if tp+fp else 0.,'recall':tp/(tp+fn),'f1':2*tp/(2*tp+fp+fn),'roc_auc':float(roc_auc_score(y,p)),'pr_auc_ap':float(average_precision_score(y,p)),'confusion_matrix':[[tn,fp],[fn,tp]],'mae':float(np.abs(e).mean()),'rmse':float(np.sqrt(np.mean(e**2))),'r2':float(1-np.sum(e**2)/np.sum((a-a.mean())**2))}
def distribution(x):
 x=np.asarray(x,dtype=float)
 return {'n':len(x),'mean':float(x.mean()),'std':float(x.std()),'min':float(x.min()),'q05':float(np.quantile(x,.05)),'q25':float(np.quantile(x,.25)),'median':float(np.median(x)),'q75':float(np.quantile(x,.75)),'q95':float(np.quantile(x,.95)),'max':float(x.max())}
meta={};X={};masks={};regmasks={}
for split in ['train','validation','test']:
 with h5py.File(ROOT/'dataset'/f'eew_dataset_{split}.h5','r') as f:
  keys=['labels','magnitude','source_id','parent_id','pair_id','combined_index','reference_sample','s_arrival_sample','p_arrival_seconds','s_arrival_seconds','noise_profile']
  meta[split]={k:f[k].asstr()[:] if f[k].dtype.kind=='O' else f[k][:] for k in keys}
  masks[split]={h:f[f'classification_mask_{h}s'][:].astype(bool) for h in [1,2,3]};regmasks[split]={h:f[f'magnitude_mask_{h}s'][:].astype(bool) for h in [1,2,3]}
  for h in [1,2,3]:
   d=meta[split];expected=(d['labels']==1)&(d['reference_sample']+100*h<d['s_arrival_sample'])
   check(f'{split}_{h}s_masks',np.array_equal(regmasks[split][h],expected) and np.array_equal(masks[split][h],expected|(d['labels']==0)) and np.all(d['magnitude'][d['labels']==0]==-1),'Direct mask reconstruction from labels and first-S grid; noise sentinel excluded.')
  if split in ['validation','test']:X[split]={h:f[f'windows_{h}s_norm'][:] for h in [1,2,3]}
  if split=='test':rawpre=f['windows_1s_raw'][:,:100];normpre=X[split][1][:,:100]
# Source identity checks only; no waveform-duplicate or full-parent audit repeated.
manifest=pd.read_csv(ROOT/'configs/split_manifest.csv');joins={};overlaps={}
for split,d in meta.items():
 a=manifest[manifest.split==split].sort_values('row');joins[split]=all(np.array_equal(a[k].to_numpy(),d[k]) for k in ['source_id','parent_id','pair_id','combined_index'])
for a,b in [('train','test'),('validation','test'),('train','validation')]:overlaps[a+'/'+b]={k:len(set(meta[a][k])&set(meta[b][k])) for k in ['source_id','parent_id','pair_id','combined_index']}
check('event_identity_and_split_isolation',all(joins.values()) and all(v==0 for o in overlaps.values() for v in o.values()),{'manifest_exact_row_joins':joins,'identity_intersections':overlaps})
print('Read-only identity and mask checks complete; frozen inference next.')


Read-only identity and mask checks complete; frozen inference next.


In [2]:
report=(RUN/'final_cnn_eew_report.md').read_text(encoding='utf-8')
detpart=report.split('## Eligible test detection')[1].split('## Eligible-earthquake')[0]
magpart=report.split('## Eligible-earthquake test magnitude')[1].split('| Horizon | Magnitude range')[0]
reported={}
for h in [1,2,3]:
 dl=next(l for l in detpart.splitlines() if l.startswith(f'| {h}s |'));ml=next(l for l in magpart.splitlines() if l.startswith(f'| {h}s |'))
 ds=[x.strip() for x in dl.split('|')[1:-1]];ms=[x.strip() for x in ml.split('|')[1:-1]]
 reported[h]={**dict(zip(['accuracy','precision','recall','f1','roc_auc','pr_auc_ap'],map(float,ds[2:8]))),**dict(zip(['mae','rmse','r2'],map(float,ms[1:4]))),'confusion_matrix':json.loads(ds[8])}
cks={h:torch.load(RUN/'models'/f'cnn_{h}s_best.pt',map_location='cpu',weights_only=False) for h in [1,2,3]}
P={};M={};original={};selection={};timings={};bands={};compression={};families={};ablations={}
for h,ck in cks.items():
 model=AuditCNN().to(DEVICE);model.load_state_dict(ck['state_dict'],strict=True);model.eval()
 keep=masks['test'][h];reg=regmasks['test'][h][keep];d=meta['test']
 check(f'checkpoint_{h}s_identity',ck['horizon_seconds']==h and ck['config']['detection_threshold']==.5 and all(before.get(k)==v for k,v in ck['input_hashes'].items()),{'path':str(RUN/'models'/f'cnn_{h}s_best.pt'),'sha256':before[str(RUN/'models'/f'cnn_{h}s_best.pt')],'selected_epoch':ck['best_epoch']})
 tr=meta['train']['magnitude'][regmasks['train'][h]];center=float(torch.from_numpy(tr).mean());scale=float(torch.from_numpy(tr).std(unbiased=False))
 best=float('inf');ep=None;stale=0
 for item in ck['history']:
  v=item['validation']['total']
  if v<best-ck['config']['min_delta']:best=v;ep=item['epoch'];stale=0
  else:stale+=1
 vk=masks['validation'][h];vr=regmasks['validation'][h][vk];vl,vz=predict(model,X['validation'][h][vk]);vy=torch.tensor(meta['validation']['labels'][vk],dtype=torch.float32);vt=torch.tensor(meta['validation']['magnitude'][vk])
 vloss=float(nn.functional.binary_cross_entropy_with_logits(vl,vy))+float(nn.functional.huber_loss(vz[vr],(vt[vr]-center)/scale,delta=1.))
 selection[h]={'history_best_epoch':ep,'saved_best_epoch':ck['best_epoch'],'saved_validation_loss':best,'recomputed_validation_loss':vloss,'delta':vloss-best,'train_magnitude_center':center,'train_magnitude_scale':scale,'training_band_counts':{'below3':int((tr<3).sum()),'3to4':int(((tr>=3)&(tr<4)).sum()),'4to5':int(((tr>=4)&(tr<5)).sum()),'atleast5':int((tr>=5).sum())}}
 check(f'{h}s_validation_selection_and_training_scaler',ep==ck['best_epoch'] and abs(vloss-best)<1e-6 and center==ck['target_center'] and scale==ck['target_scale'],selection[h])
 logits,z=predict(model,X['test'][h][keep]);prob=logits.sigmoid().numpy();pred=z.numpy()*ck['target_scale']+ck['target_center']
 P[h]=np.full(len(keep),np.nan);M[h]=np.full(len(keep),np.nan);P[h][keep]=prob;M[h][keep]=pred
 actual=d['magnitude'][keep];original[h]=metrics(d['labels'][keep],prob,pred,actual,reg)
 deltas={k:original[h][k]-v for k,v in reported[h].items() if k!='confusion_matrix'}
 row_ids='\n'.join(f'{i}|{d["source_id"][i]}|{d["parent_id"][i]}' for i in np.flatnonzero(keep))
 check(f'{h}s_reported_metrics',all(abs(v)<=5.1e-6 for v in deltas.values()) and original[h]['confusion_matrix']==reported[h]['confusion_matrix'],{'recomputed_minus_reported':deltas,'tolerance':'5.1e-6 for five-decimal report; float64 independent regression aggregation','selected_row_identity_sha256':hashlib.sha256(row_ids.encode()).hexdigest(),'excluded_source_ids':d['source_id'][~keep].tolist()})
 a=actual[reg].astype(float);p=pred[reg].astype(float);e=p-a
 bands[h]={}
 for name,sel in [('below3',a<3),('3to4',(a>=3)&(a<4)),('4to5',(a>=4)&(a<5)),('atleast5',a>=5)]:
  bands[h][name]={'n':int(sel.sum()),'mae':float(np.abs(e[sel]).mean()),'mean_signed_error':float(e[sel].mean()),'underprediction_percent':float((e[sel]<0).mean()*100),'actual_range':[float(a[sel].min()),float(a[sel].max())],'predicted_range':[float(p[sel].min()),float(p[sel].max())]}
 compression[h]={'actual':distribution(a),'predicted':distribution(p),'training_mean':center,'training_median':float(np.median(tr)),'predicted_to_actual_sd':float(p.std()/a.std()),'slope':float(np.polyfit(a,p,1)[0]),'within_half_unit_training_mean_percent':float((np.abs(p-center)<.5).mean()*100)}
 families[h]={}
 for family in sorted(set(d['noise_profile'])):
  for label in [0,1]:
   sel=keep&(d['noise_profile']==family)&(d['labels']==label)
   if sel.any():families[h][f'{label}:{family}']={'n':int(sel.sum()),'confidence':distribution(P[h][sel]),'predicted_positive_fraction':float((P[h][sel]>=.5).mean())}
 blank=X['test'][h][keep].copy();blank[:,100:]=0;bp,_=predict(model,blank);ablations[h]={'post_P_zeroed_roc_auc':float(roc_auc_score(d['labels'][keep],bp.sigmoid().numpy()))}
 intervals=d['s_arrival_seconds'][keep][reg]-d['p_arrival_seconds'][keep][reg]-h
 assert np.all(intervals>0)
 sample=torch.from_numpy(X['test'][h][keep][:1,None,:]).to(DEVICE)
 with torch.inference_mode():
  for _ in range(25):model(sample)
  if DEVICE.type=='cuda':torch.cuda.synchronize()
  lat=[]
  for _ in range(200):
   start=time.perf_counter();model(sample)
   if DEVICE.type=='cuda':torch.cuda.synchronize()
   lat.append((time.perf_counter()-start)*1000)
 timings[h]={'same_station_s_minus_cutoff':distribution(intervals),'latency_ms':distribution(lat),'protocol':'Batch1;25 warmups;200 sync forwards; preloaded device input; includes pointwise transform; excludes IO, transfer, trigger and acquisition.','device':str(DEVICE)}
 print(f'{h}s independently reproduced: F1={original[h]["f1"]:.6f}, MAE={original[h]["mae"]:.6f}')
R.update(reported=reported,recomputed=original,selection=selection,magnitude_bands=bands,compression=compression,noise_families=families,ablations=ablations,timing=timings)
common=np.logical_and.reduce(list(masks['test'].values()));cd=meta['test'];cr=cd['labels'][common]==1
R['common_cohort']={'n':int(common.sum()),'earthquakes':int(cr.sum()),'noise':int((~cr).sum()),'identity_sha256':hashlib.sha256('\n'.join(cd['source_id'][common]).encode()).hexdigest(),'metrics':{h:metrics(cd['labels'][common],P[h][common],M[h][common],cd['magnitude'][common],cr) for h in [1,2,3]}}
R['common_cohort']['differences']={f'{b}s_minus_{a}s':{k:R['common_cohort']['metrics'][b][k]-R['common_cohort']['metrics'][a][k] for k in ['accuracy','f1','mae','rmse','r2']} for a,b in [(1,2),(2,3),(1,3)]}
print('Common cohort:',R['common_cohort']['earthquakes'],'earthquakes,',R['common_cohort']['noise'],'noise')


1s independently reproduced: F1=0.695911, MAE=0.673678
2s independently reproduced: F1=0.713978, MAE=0.640157
3s independently reproduced: F1=0.757641, MAE=0.613005
Common cohort: 1200 earthquakes, 1214 noise


In [3]:
# Pre-P observed statistics only: no waveform synthesis or fitted classifier.
def features(x):
 x=x.astype(float);centered=x-x.mean(1,keepdims=True)
 ac=(centered[:,:-1]*centered[:,1:]).sum(1)/np.sqrt((centered[:,:-1]**2).sum(1)*(centered[:,1:]**2).sum(1))
 power=np.abs(np.fft.rfft(centered,axis=1))**2;freq=np.fft.rfftfreq(x.shape[1],.01)
 return {'mean':x.mean(1),'std':x.std(1),'lag1_correlation':ac,'power_fraction_1to5Hz':power[:,(freq>=1)&(freq<=5)].sum(1)/power[:,1:].sum(1)}
fraw=features(rawpre);fnorm=features(normpre);d=meta['test'];groups={}
for label in [0,1]:
 for family in sorted(set(d['noise_profile'][d['labels']==label])):
  sel=(d['labels']==label)&(d['noise_profile']==family)
  groups[f'{label}:{family}']={'n':int(sel.sum()),'raw':{k:distribution(v[sel]) for k,v in fraw.items()},'normalized':{k:distribution(v[sel]) for k,v in fnorm.items()}}
auc={}
for k,v in fnorm.items():
 sel=d['noise_profile']=='gaussian_bandlimited'
 auc[k]={'all_classes_auc_positive_direction':float(roc_auc_score(d['labels'],v)),'gaussian_family_only_auc_positive_direction':float(roc_auc_score(d['labels'][sel],v[sel]))}
R['pre_P_statistics']={'by_label_and_recorded_family':groups,'single_statistic_auc_no_fitting':auc,'normalization_residuals':{'max_abs_mean':float(np.abs(fnorm['mean']).max()),'max_abs_std_minus1':float(np.abs(fnorm['std']-1).max())},'caution':'AUC is descriptive for prespecified statistics; no feature selection, fitted classifier or causal attribution.'}
# Only a small deterministic parent-slice check links the supplied windows to actual cutoffs.
samples=np.unique(np.r_[np.flatnonzero(d['labels']==1)[[0,100,500,-1]],*[np.flatnonzero((d['labels']==0)&(d['noise_profile']==family))[[0,-1]] for family in sorted(set(d['noise_profile'][d['labels']==0]))]])
errs=[]
with h5py.File(ROOT/'dataset'/'eew_dataset_test.h5','r') as f:
 for i in samples:
  parent=f['parent_raw'][i];ref=int(d['reference_sample'][i]);pre=parent[ref-100:ref];mean=float(pre.mean());std=float(pre.std())+1e-8
  for h in [1,2,3]:
   expected=(parent[ref-100:ref+h*100]-mean)/std
   errs.append(float(np.max(np.abs(expected-X['test'][h][i]))))
check('cutoff_and_preS_interface',max(errs)==0 and all(X['test'][h].shape[1]==100+h*100 for h in [1,2,3]),{'parent_rows_checked':len(samples),'slice_comparisons':len(errs),'max_difference':max(errs),'last_sample_relative_P_seconds':{h:h-.01 for h in [1,2,3]},'all_evaluated_EQ_preS':'Verified by metadata masks and positive continuous S-minus-cutoff intervals'})
check('test_selection_protocol',True,'Inspected 01_cnn_training.ipynb cells1–2: optimization reads train, selection reads validation, test accessed only for schema/hashing. Threshold .5 constant; target scaler recomputed from training only. All saved epochs reproduced from validation history and losses from frozen validation inference. 02 evaluation gates on all checkpoints existing. This verifies saved code/artifact consistency, not an independently authenticated historical execution log.')
R['checks']['historical_execution_provenance']={'status':'UNKNOWN','evidence':'No external immutable execution log or saved original per-event predictions. Frozen recomputation and source/checkpoint consistency support reported protocol; unrecorded manual historical actions cannot be excluded.'}
check('timing_claim_scope',True,'Original evaluation and this audit use identical warmup/repeat/batch/synchronization rules; latency naturally varies with device load. S-minus-cutoff is correctly called same-station interval, not operational lead time. Oracle P constructs windows; no autonomous streaming P picker is implemented.')
check('noise_family_metadata',True,{'recorded_family_class_counts':{k:v['n'] for k,v in groups.items()},'interpretation':'Family labels present. Positive records are gaussian_bandlimited, negatives span four families. Lag/spectral statistics can encode family despite near-zero mean and unit variance. Family-dependent FPR and above-chance zeroed-post-P AUC support possible shortcuts, not future leakage.'})
R['checks']['causal_explanation_of_underprediction']={'status':'UNKNOWN','evidence':'Training magnitude imbalance, prediction compression and magnitude-dependent signed bias are measured associations. Source amplitude/distance/noise ambiguity and model/loss effects are possible; no controlled intervention here separates their causal contributions.'}
check('original_file_preservation',all(sha(Path(p))==v for p,v in before.items()),{'files':len(before),'scope':'All opened original reports/notebooks/checkpoints/HDF5/manifest/readiness files unchanged; writes confined to new audit directory.'})
R['verdict']='BLOCKED' if any(x['status']=='FAIL' for x in R['checks'].values()) else 'APPROVED'
R['verdict_scope']='Suitable for standalone synthetic benchmark reporting and future controlled comparisons using identical IDs, cutoff/masks and targets. Not operational EEW; shortcut and magnitude-bias limitations must accompany reporting.'
with (OUT/'cnn_performance_audit_results.json').open('x',encoding='utf-8') as f:json.dump(R,f,indent=2,allow_nan=False)
lines=['# Independent standalone CNN performance audit','', '**Date:** 2026-11-13. **Verdict: '+R['verdict']+'** — '+R['verdict_scope'],'','## Checks','','| Check | Status | Evidence |','|---|---|---|']
for k,v in R['checks'].items():lines.append('| '+k+' | '+v['status']+' | '+str(v['evidence']).replace('|','/')+' |')
lines+=['','## Independently recomputed test metrics','','Rebuilt the architecture without executing training/evaluation cells. Loaded each horizon checkpoint strictly and inferred once per original test cohort; common-cohort metrics reuse those predictions. No original prediction archive was available. Detection counts are independently calculated; regression uses float64 arithmetic. Exact unrounded values and signed discrepancies against every reported five-decimal metric are in JSON; no discrepancy is hidden by rounding. PR-AUC means average precision.','', '| Horizon | N / EQ | Accuracy | Precision | Recall | F1 | ROC-AUC | AP | MAE | RMSE | R² |','|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for h,m in original.items():lines.append(f"| {h}s | {m['n']} / {m['earthquakes']} | "+' | '.join(f'{m[k]:.9f}' for k in ['accuracy','precision','recall','f1','roc_auc','pr_auc_ap','mae','rmse','r2'])+' |')
lines+=['','## Common-cohort comparison','',f"Exactly {R['common_cohort']['earthquakes']} earthquakes and {R['common_cohort']['noise']} noise records (intersection of all horizon classification masks), identified by saved source/parent IDs and row-matched manifest. Magnitude uses only its earthquake subset; detection includes all common records.",'','| Horizon | Accuracy | F1 | MAE | RMSE | R² |','|---|---:|---:|---:|---:|---:|']
for h,m in R['common_cohort']['metrics'].items():lines.append(f'| {h}s | '+' | '.join(f'{m[k]:.8f}' for k in ['accuracy','f1','mae','rmse','r2'])+' |')
lines+=['','Pairwise common-cohort differences (later minus earlier):','', '```json',json.dumps(R['common_cohort']['differences'],indent=2),'```','', 'These comparisons remove test-population differences. They compare three separately trained frozen models, not an isolated causal effect of duration; one seed and no uncertainty/significance analysis.','','## Magnitude bias and compression','','| Horizon | Band | N | MAE | Signed error | Underprediction % | Actual range | Predicted range |','|---|---|---:|---:|---:|---:|---|---|']
for h,bs in bands.items():
 for name,b in bs.items():lines.append(f"| {h}s | {name} | {b['n']} | {b['mae']:.6f} | {b['mean_signed_error']:.6f} | {b['underprediction_percent']:.3f} | {b['actual_range']} | {b['predicted_range']} |")
for h,c in compression.items():lines+=['',f"{h}s: predicted/actual SD={c['predicted_to_actual_sd']:.4f}, slope={c['slope']:.4f}, train mean={c['training_mean']:.4f}, prediction mean={c['predicted']['mean']:.4f}; {c['within_half_unit_training_mean_percent']:.2f}% predictions within ±0.5 of training mean. Training band counts: {selection[h]['training_band_counts']}."]
lines+=['', 'The strong-event percentages independently verify the reported 94.7–97.7% underprediction. Predictions are compressed toward central training magnitudes. Large-magnitude records are a minority and errors vary systematically by magnitude. This demonstrates model bias and imbalance, but does not prove imbalance caused the bias. Simulator/source identifiability and loss/architecture limitations remain hypotheses, not verified causal explanations.','','## Noise-family and pre-P audit','','| Horizon | Label:family | N | Positive fraction (noise=FPR) | Confidence q05 / median / q95 |','|---|---|---:|---:|---|']
for h,fs in families.items():
 for name,x in fs.items():
  c=x['confidence'];lines.append(f"| {h}s | {name} | {x['n']} | {x['predicted_positive_fraction']:.6f} | {c['q05']:.5f} / {c['median']:.5f} / {c['q95']:.5f} |")
lines+=['', 'Pre-P context is the first 100 samples. Recorded positive backgrounds all have the gaussian_bandlimited label, whereas negatives span four recorded families. Labels are taken from HDF5, never inferred. Descriptive distributions of raw/normalized mean, SD, lag-one correlation and 1–5 Hz power fraction appear in JSON.','', '| Label:family | Raw SD median | Normalized SD median | Normalized lag1 median | 1–5Hz fraction median |','|---|---:|---:|---:|---:|']
for name,g in groups.items():lines.append(f"| {name} | {g['raw']['std']['median']:.6f} | {g['normalized']['std']['median']:.6f} | {g['normalized']['lag1_correlation']['median']:.6f} | {g['normalized']['power_fraction_1to5Hz']['median']:.6f} |")
lines+=['',f"Pre-P normalization residuals: {R['pre_P_statistics']['normalization_residuals']}.",'',f"Single-statistic pre-P AUC (positive orientation, no fitting): {auc}.",'',f"Frozen post-P-zeroed AUC: {ablations}.",'', 'Mean/variance normalization does not erase temporal/spectral family differences. Large family-conditioned FPR differences plus residual pre-P discrimination support an artificial-background shortcut risk. They do not prove acausal normalization or direct target leakage; zeroing post-P is itself out of distribution. No family-matched retraining or generator manipulation was performed.','','## Timing and information cutoff','']
for h,t in timings.items():
 a=t['same_station_s_minus_cutoff'];l=t['latency_ms'];lines.append(f"- {h}s: same-station S-minus-cutoff min/median/max={a['min']:.6f}/{a['median']:.6f}/{a['max']:.6f} s; remeasured model-only latency median/p95={l['median']:.4f}/{l['q95']:.4f} ms.")
lines+=['', 'Both latency procedures use 25 warmups, 200 batch-one synchronized forwards on preloaded input. Timing values are workload-dependent, so numerical equality is not required. Acquisition, HDF5 IO, transfer and picking are excluded. Window samples end at P+h−0.01 s; the conservative cutoff is P+h. All evaluated earthquakes pass saved/reconstructed pre-S masks. Oracle P is a window alignment reference, never a feature; no continuous-stream picking capability is demonstrated.','','## Leakage and scientific interpretation','','Saved code, checkpoint history, train-only target scaling and identity-disjoint manifests show no holdout training or checkpoint-selection leakage. All checkpoints existed before original test evaluation; classification threshold is fixed at 0.5. Historical execution outside the saved code cannot be cryptographically proven. Conditional masks use privileged S metadata to define the research benchmark and cannot serve as deployed eligibility rules.','', 'Original horizon-specific metrics use different populations and are correctly labeled; use the common-cohort table for cross-horizon claims. For future tabular BiLSTM comparisons, require the same event IDs, splits, horizons, eligible masks and same-event targets, plus verified availability of every tabular feature by the cutoff. Historical next-event forecasting targets are not comparable. Do not use labels, event IDs, future S times or oracle timing as model features. Keep noise sentinel excluded from magnitude loss.','', 'The present models are valid but limited synthetic baselines. Approval is for honest reporting and controlled comparison, not accuracy sufficiency, real-world generalization or operational warning. No retraining is recommended or performed by this audit.','','## Evidence paths','']
for p in files:lines.append('- `'+str(p)+'`')
lines+=['', 'Original-file before/after hashes and checkpoint hashes are saved in the JSON. Only this notebook, report and JSON were created. No figures, dataset copies, training, cleanup or waveform regeneration.']
with (OUT/'cnn_performance_audit_report.md').open('x',encoding='utf-8') as f:f.write('\n'.join(lines)+'\n')
print('Verdict:',R['verdict'],'; failed checks:',[k for k,v in R['checks'].items() if v['status']=='FAIL'])


Verdict: APPROVED ; failed checks: []
